# Recombination with GRG

**Note:** To add sample nodes (e.g., offspring from recombination), use grgl from the `grg_modify_improve` branch:

```bash
pip install wheel
```

```bash
git clone -b grg_modify_improve --recursive https://github.com/aprilweilab/grgl.git
cd grgl && python setup.py bdist_wheel && pip install --force-reinstall dist/*.whl
```

The `MutableGRG.set_samples(sample_nodes)` API lets you add new sample nodes by passing the full list of sample node IDs (including the new offspring).

In [25]:
import numpy as np
import bisect

NEGATIVE_NODE_IDS = []  
def get_breakpoints(N, p=0.1, min_breakpoints=3):
    """
    Return breakpoint positions. Ensures at least min_breakpoints in interior
    so recombination is less likely to be entire interval from one parent.
    """
    bp = np.where(np.random.binomial(1, p, N))[0]
    # Ensure at least one interior breakpoint to avoid entire-interval inheritance
    interior = np.arange(1, N)
    while len(bp) < min_breakpoints and len(interior) >= min_breakpoints:
        bp = np.unique(np.concatenate([bp, np.random.choice(interior, min(min_breakpoints, len(interior)), replace=False)]))
        bp = np.sort(bp)
    print(f"Generated breakpoints: {bp}")
    return bp

def recombination_intervals(h1, h2, N):
    """
    Returns list of segments: [(source_parent_id, end_coord), ...]
    """
    bp = get_breakpoints(N)
    start = np.random.binomial(1, 0.5, 1)[0]
    parents = [h1, h2]
    
    segments = []
    for i, K in enumerate(bp):
        segments.append((parents[(start + i) % 2], K))
    segments.append((parents[(start + len(bp)) % 2], N))
    return segments

In [26]:
from pygrgl import MutableGRG, Mutation, save_grg, grg_to_cyto_json
import pygrgl

def create_simple_grg():
    grg = MutableGRG(4, 1, True) 
    
    # Create 4 internal nodes (4, 5, 6, 7)
    grg.make_node()  # node 4 (has m6, m7)
    grg.make_node()  # node 5 (has m8)
    grg.make_node()  # node 6 (has m9)
    grg.make_node()  # node 7 (has m10, m11)
    
    # Add edges (parent -> child) 
    grg.connect(6, 0)  # 6 -> 0
    grg.connect(6, 5)  # 6 -> 5
    grg.connect(7, 5)  # 7 -> 5
    grg.connect(7, 4)  # 7 -> 4
    grg.connect(5, 1)  # 5 -> 1
    grg.connect(5, 4)  # 5 -> 4
    grg.connect(4, 1)  # 4 -> 1
    grg.connect(4, 2)  # 4 -> 2
    grg.connect(4, 3)  # 4 -> 3
    
    # Add mutations (m1-m11)
    grg.add_mutation(Mutation(0, "A", "G"), 1)   # m1 on node 1
    grg.add_mutation(Mutation(1, "C", "T"), 0)   # m2 on node 0
    grg.add_mutation(Mutation(2, "G", "A"), 0)   # m3 on node 0
    grg.add_mutation(Mutation(3, "T", "C"), 2)   # m4 on node 2
    grg.add_mutation(Mutation(4, "A", "T"), 3)   # m5 on node 3
    grg.add_mutation(Mutation(5, "C", "G"), 4)   # m6 on node 4
    grg.add_mutation(Mutation(6, "G", "C"), 4)   # m7 on node 4
    grg.add_mutation(Mutation(7, "T", "A"), 5)   # m8 on node 5
    grg.add_mutation(Mutation(8, "A", "C"), 6)   # m9 on node 6
    grg.add_mutation(Mutation(9, "C", "A"), 7)  # m10 on node 7
    grg.add_mutation(Mutation(10, "G", "T"), 7)  # m11 on node 7
    
    return grg

# Create and save the GRG
simple_grg = create_simple_grg()


print("=== Simple GRG Created ===")
print(f"Nodes: {simple_grg.num_nodes}")
print(f"Edges: {simple_grg.num_edges}")
print(f"Mutations: {simple_grg.num_mutations}")
print(f"Samples: {simple_grg.get_sample_nodes()}")
print()

# Display structure
print("Node details:")
for node_id in range(simple_grg.num_nodes):
    parents = simple_grg.get_up_edges(node_id)
    children = simple_grg.get_down_edges(node_id)
    muts = simple_grg.get_mutations_for_node(node_id)
    mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
    is_sample = "[SAMPLE]" if simple_grg.is_sample(node_id) else ""
    print(f"  Node {node_id}: parents={parents}, children={children}, muts={mut_names} {is_sample}")

# Save to file
save_grg(simple_grg, "simple_example.grg")
print()
print("Saved to simple_example.grg")

cyto_data = grg_to_cyto_json(simple_grg)
print()
print("Cytoscape JSON representation:")
print(f"  Nodes: {len(cyto_data['nodes'])}")
print(f"  Edges: {len(cyto_data['edges'])}")
for node in cyto_data['nodes']:
    print(f"    {node['data']}")
for edge in cyto_data['edges']:
    print(f"    {edge['data']}")

=== Simple GRG Created ===
Nodes: 8
Edges: 9
Mutations: 11
Samples: [0, 1, 2, 3]

Node details:
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 3: parents=[4], children=[], muts=['m5'] [SAMPLE]
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7'] 
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8'] 
  Node 6: parents=[], children=[0, 5], muts=['m9'] 
  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11'] 

Saved to simple_example.grg

Cytoscape JSON representation:
  Nodes: 8
  Edges: 9
    {'id': 'n0', 'label': 'id=0, S, mutations({1, 2})', 'is_sample': 'True'}
    {'id': 'n1', 'label': 'id=1, S, mutations({0})', 'is_sample': 'True'}
    {'id': 'n2', 'label': 'id=2, S, mutations({3})', 'is_sample': 'True'}
    {'id': 'n3', 'label': 'id=3, S, mutations({4})', 'is_sample': 'True'}
    {'id': 'n4', 'label': 'id=4, mutations({5, 6})',

In [18]:
GRG_FILE = "simple_example.grg" 

In [19]:
class NonDuplicationRecombination:
    """
    Non-duplication GRG recombination algorithm.

    Optimizations vs. the recursive version:
    - Iterative DFS for `_recurse_attach` and `_get_node_and_ancestor_span`
      (avoids Python's recursion limit on deep GRGs).
    - O(1) offspring reverse lookup via a dict alongside NEGATIVE_NODE_IDS.
    - Per-recombiner cache for `get_up_edges()`.
    - Generation-counter arrays replace per-call Python sets for
      `visited` (per DFS) and `connected` (per offspring).
    - Optional deferred sample updates via `defer_sample_updates`.
    - Hot-loop micro-optimizations: cache-hit fast paths for mutation
      range and ancestral coverage are inlined into `_recurse_attach`,
      with class attributes hoisted into local variables (CPython
      optimizes LOAD_FAST faster than LOAD_ATTR).
    - `_extract_bubble` no longer invalidates the caches of node_id's
      parents; bubble extraction does not affect their span or ancestral
      coverage, so invalidation just thrashes high-traffic cache entries.
    - Per-parent interval batching: `recombine_multi` groups all segments
      contributed by the same parent into a single traversal whose query
      is the union of those intervals. This avoids re-bubbling the same
      ancestor when one parent contributes mutations across multiple
      disjoint segments of the offspring's genome.
    - `allow_sort=False` on `get_mutations_for_node`: with the updated
      grgl API, adding mutations only to newly-created bubble nodes
      preserves the mutation-to-node sort flag. Reading with
      `allow_sort=False` acts as a tripwire: any future change that adds
      a mutation to an older node will throw rather than silently
      trigger an O(N log N) resort.
    """

    debug_mode = False

    def __init__(self, grg):
        self.grg = grg
        self.genome_length = grg.bp_range[1]
        self.original_bp_range = grg.bp_range
        self.NEGATIVE_NODE_IDS = []
        self._negative_node_index = {}
        self._modified_nodes = set()
        self._pending_bubbles = []
        self._pending_sample_removals = set()

        self.defer_sample_updates = False

        self._mutation_cache = {}
        self._pos_cache = {}
        self._up_edges_cache = {}

        self.span_cache = [False] * self.grg.num_nodes
        self.anc_cov_cache = [False] * self.grg.num_nodes

        self._visited_gen = [0] * self.grg.num_nodes
        self._connected_gen = [0] * self.grg.num_nodes
        self._gen_visited = 0
        self._gen_connected = 0

    # ------------------------------------------------------------------
    # Per-node array growth
    # ------------------------------------------------------------------

    def _grow_node_arrays(self, node_id):
        target = node_id + 1
        n = len(self.span_cache)
        if n < target:
            pad = target - n
            self.span_cache.extend([False] * pad)
            self.anc_cov_cache.extend([False] * pad)
        n = len(self._visited_gen)
        if n < target:
            pad = target - n
            self._visited_gen.extend([0] * pad)
            self._connected_gen.extend([0] * pad)

    def _sync_to_grg(self):
        n = self.grg.num_nodes
        if n > 0:
            self._grow_node_arrays(n - 1)

    # ------------------------------------------------------------------
    # Cached graph access
    # ------------------------------------------------------------------

    def _get_up_edges_cached(self, node_id):
        cached = self._up_edges_cache.get(node_id)
        if cached is None:
            cached = list(self.grg.get_up_edges(node_id))
            self._up_edges_cache[node_id] = cached
        return cached

    # ------------------------------------------------------------------
    # Mutation caches
    # ------------------------------------------------------------------

    def _get_node_mutations(self, node_id):
        if node_id not in self._mutation_cache:
            mut_ids = self.grg.get_mutations_for_node(node_id, allow_sort=False)
            mutations = []
            for mut_id in mut_ids:
                mut = self.grg.get_mutation_by_id(mut_id)
                mutations.append((mut_id, mut.position))
            combined = sorted(mutations, key=lambda x: x[1])
            self._mutation_cache[node_id] = combined
            self._pos_cache[node_id] = [m[1] for m in combined]
        return self._mutation_cache[node_id]

    def _get_mutation_range(self, node_id, L, R):
        self._get_node_mutations(node_id)
        positions = self._pos_cache[node_id]
        if not positions:
            return 0, 0
        return bisect.bisect_left(positions, L), bisect.bisect_left(positions, R)

    # ------------------------------------------------------------------
    # Interval-union helpers
    # ------------------------------------------------------------------
    # `intervals` is always a sorted tuple of disjoint, coalesced,
    # half-open (L, R) pairs representing the union of genomic regions
    # the offspring still needs to inherit on the current traversal.

    @staticmethod
    def _intervals_disjoint_from(intervals, lo, hi):
        for L, R in intervals:
            if L >= hi:
                return True
            if R > lo:
                return False
        return True

    @staticmethod
    def _intervals_contain(intervals, lo, hi):
        # True iff some single piece (L_i, R_i) of the union has
        # L_i <= lo and hi <= R_i. Because the pieces are disjoint and
        # coalesced, no half-open [lo, hi) can be covered by spanning
        # multiple pieces -- coalescing collapses adjacency.
        for L, R in intervals:
            if L > lo:
                return False
            if R >= hi:
                return True
        return False

    @staticmethod
    def _clip_intervals(intervals, lo, hi):
        result = []
        for L, R in intervals:
            if R <= lo:
                continue
            if L >= hi:
                break
            nL = L if L > lo else lo
            nR = R if R < hi else hi
            result.append((nL, nR))
        return tuple(result)

    # ------------------------------------------------------------------
    # Iterative span / ancestral coverage
    # ------------------------------------------------------------------

    def _get_node_and_ancestor_span(self, node_id):
        if self.span_cache[node_id] is not False:
            return self.span_cache[node_id]

        stack = [(node_id, False)]
        scheduled = {node_id}

        while stack:
            nid, processed = stack.pop()

            if processed:
                min_p, max_p = float('inf'), float('-inf')
                node_muts = self._get_node_mutations(nid)
                if node_muts:
                    min_p = node_muts[0][1]
                    max_p = node_muts[-1][1]
                for parent in self._get_up_edges_cached(nid):
                    anc = self.span_cache[parent]
                    if anc:
                        if anc[0] < min_p: min_p = anc[0]
                        if anc[1] > max_p: max_p = anc[1]
                self.span_cache[nid] = None if min_p == float('inf') else (min_p, max_p)
                scheduled.discard(nid)
            else:
                stack.append((nid, True))
                for parent in self._get_up_edges_cached(nid):
                    if self.span_cache[parent] is False and parent not in scheduled:
                        stack.append((parent, False))
                        scheduled.add(parent)

        return self.span_cache[node_id]

    def _get_ancestral_coverage(self, node_id):
        if self.anc_cov_cache[node_id] is not False:
            return self.anc_cov_cache[node_id]

        parents = self._get_up_edges_cached(node_id)
        if not parents:
            self.anc_cov_cache[node_id] = None
            return None

        min_pos, max_pos = float('inf'), float('-inf')
        for parent in parents:
            p_span = self._get_node_and_ancestor_span(parent)
            if p_span:
                if p_span[0] < min_pos: min_pos = p_span[0]
                if p_span[1] > max_pos: max_pos = p_span[1]

        result = None if min_pos == float('inf') else (min_pos, max_pos + 1)
        self.anc_cov_cache[node_id] = result
        return result

    # ------------------------------------------------------------------
    # Bubble extraction
    # ------------------------------------------------------------------

    def _extract_bubble(self, node_id, relevant_mut_ids, offspring_id, intervals):
        bubble_id = self.grg.make_node()
        self._grow_node_arrays(bubble_id)

        self.grg.connect(bubble_id, node_id)
        self.grg.connect(bubble_id, -offspring_id)

        # node_id just gained a new up-edge; drop its cached up-edges list.
        self._up_edges_cache.pop(node_id, None)

        self._pending_bubbles.append({
            'node_id': node_id,
            'bubble_id': bubble_id,
            'relevant_mut_ids': relevant_mut_ids,
        })

        # Track only nodes whose own caches actually become stale:
        # node_id loses mutations (mutation/pos/anc_cov are stale) and the
        # new bubble_id needs a clean slot. node_id's parents are NOT
        # invalidated -- their span and anc_cov depend only on themselves
        # and their own ancestors, neither of which changes here.
        self._modified_nodes.add(node_id)
        self._modified_nodes.add(bubble_id)

        return bubble_id

    # ------------------------------------------------------------------
    # Iterative recurse-attach (hot path)
    # ------------------------------------------------------------------

    def _recurse_attach(self, root_id, offspring_id, intervals):
        """`intervals`: sorted, coalesced tuple of disjoint half-open (L, R) pairs."""
        if not intervals:
            return

        self._gen_visited += 1
        gen_v = self._gen_visited
        gen_c = self._gen_connected

        # Hoist attributes / globals into locals for the inner loop.
        # CPython's LOAD_FAST is faster than LOAD_ATTR; over hundreds of
        # thousands of iterations these adds up.
        visited_gen = self._visited_gen
        connected_gen = self._connected_gen
        pos_cache = self._pos_cache
        mutation_cache = self._mutation_cache
        anc_cov_cache = self.anc_cov_cache
        bisect_left = bisect.bisect_left
        get_node_mutations = self._get_node_mutations
        get_ancestral_coverage = self._get_ancestral_coverage
        get_up_edges_cached = self._get_up_edges_cached
        extract_bubble = self._extract_bubble
        clip_intervals = NonDuplicationRecombination._clip_intervals
        grg_connect = self.grg.connect
        pending_sample_removals_add = self._pending_sample_removals.add

        neg_offspring = -offspring_id
        stack = [(root_id, intervals)]
        stack_append = stack.append
        stack_pop = stack.pop

        while stack:
            node_id, ivs = stack_pop()

            if not ivs:
                continue
            if visited_gen[node_id] == gen_v:
                continue
            visited_gen[node_id] = gen_v

            # Multi-bisect over each piece of the union; collect both the
            # count of relevant mutations and the (left, right) slice
            # bounds (only used if we end up extracting a bubble).
            positions = pos_cache.get(node_id)
            if positions is None:
                get_node_mutations(node_id)              # populates both caches
                positions = pos_cache[node_id]

            num_all = len(positions)
            num_rel = 0
            slice_bounds = None
            if num_all:
                slice_bounds = []
                for L, R in ivs:
                    left = bisect_left(positions, L)
                    right = bisect_left(positions, R)
                    if right > left:
                        slice_bounds.append((left, right))
                        num_rel += right - left

            has_all_relevant = num_all > 0 and num_rel == num_all
            has_no_relevant = num_rel == 0
            has_partial_relevant = num_rel > 0 and num_rel < num_all

            # Inlined _get_ancestral_coverage cache-hit path.
            Iu = anc_cov_cache[node_id]
            if Iu is False:
                Iu = get_ancestral_coverage(node_id)

            if Iu is None:
                if has_all_relevant and connected_gen[node_id] != gen_c:
                    grg_connect(node_id, neg_offspring)
                    connected_gen[node_id] = gen_c
                    pending_sample_removals_add(node_id)
                if has_partial_relevant:
                    muts = mutation_cache[node_id]
                    rel_mut_ids = []
                    for l, r in slice_bounds:
                        rel_mut_ids.extend(m[0] for m in muts[l:r])
                    bubble_id = extract_bubble(node_id, rel_mut_ids, offspring_id, ivs)
                    connected_gen[bubble_id] = gen_c
                continue

            Iu0 = Iu[0]
            Iu1 = Iu[1]

            # ancestral_disjoint: no piece of the union overlaps [Iu0, Iu1).
            ancestral_disjoint = True
            for L, R in ivs:
                if L >= Iu1:
                    break
                if R > Iu0:
                    ancestral_disjoint = False
                    break

            # full_coverage: some single piece of the union fully contains
            # [Iu0, Iu1). Coalesced disjoint pieces can't cover [Iu0, Iu1)
            # by spanning, so a single-piece check is sufficient.
            full_coverage = False
            for L, R in ivs:
                if L > Iu0:
                    break
                if R >= Iu1:
                    full_coverage = True
                    break

            if has_all_relevant and full_coverage:
                if connected_gen[node_id] != gen_c:
                    grg_connect(node_id, neg_offspring)
                    connected_gen[node_id] = gen_c
                    pending_sample_removals_add(node_id)
                continue

            if has_no_relevant and ancestral_disjoint:
                continue

            if has_no_relevant:
                clipped = clip_intervals(ivs, Iu0, Iu1)
                if not clipped:
                    continue
                for parent in reversed(get_up_edges_cached(node_id)):
                    stack_append((parent, clipped))
                continue

            # Partial / all relevant, not full coverage -> bubble + maybe recurse.
            muts = mutation_cache[node_id]
            rel_mut_ids = []
            for l, r in slice_bounds:
                rel_mut_ids.extend(m[0] for m in muts[l:r])
            bubble_id = extract_bubble(node_id, rel_mut_ids, offspring_id, ivs)
            connected_gen[bubble_id] = gen_c

            if not ancestral_disjoint:
                clipped = clip_intervals(ivs, Iu0, Iu1)
                if not clipped:
                    continue
                for parent in reversed(get_up_edges_cached(node_id)):
                    stack_append((parent, clipped))

    # ------------------------------------------------------------------
    # Apply deferred work, evict caches
    # ------------------------------------------------------------------

    def _apply_pending_bubbles(self):
        for bubble_op in self._pending_bubbles:
            node_id = bubble_op['node_id']
            bubble_id = bubble_op['bubble_id']
            for mut_id in bubble_op['relevant_mut_ids']:
                mut = self.grg.get_mutation_by_id(mut_id)
                self.grg.add_mutation(mut, bubble_id)
                self.grg.remove_mutation(mut_id, node_id)
        self._pending_bubbles.clear()

        if not self.defer_sample_updates:
            self.flush_sample_updates()

    def flush_sample_updates(self):
        if self._pending_sample_removals:
            current = set(self.grg.get_sample_nodes())
            current.difference_update(self._pending_sample_removals)
            self.grg.set_samples(list(current))
            self._pending_sample_removals.clear()

    def _clear_modified_caches(self):
        for node_id in self._modified_nodes:
            self._mutation_cache.pop(node_id, None)
            self._pos_cache.pop(node_id, None)
            if node_id < len(self.span_cache):
                self.span_cache[node_id] = False
                self.anc_cov_cache[node_id] = False
        self._modified_nodes.clear()

    # ------------------------------------------------------------------
    # Public entry points
    # ------------------------------------------------------------------

    def _register_offspring(self, offspring_id):
        idx = self._negative_node_index.get(offspring_id)
        if idx is None:
            idx = len(self.NEGATIVE_NODE_IDS)
            self._negative_node_index[offspring_id] = idx
            self.NEGATIVE_NODE_IDS.append(offspring_id)
        return -(idx + 1)

    def recombine(self, haplotype_A, haplotype_B, breakpoint):
        self._pending_bubbles.clear()
        self._sync_to_grg()
        offspring_id = self.grg.make_node(negative=True)
        self._grow_node_arrays(self.grg.num_nodes - 1)
        self._gen_connected += 1

        self._recurse_attach(haplotype_A, offspring_id, ((0, breakpoint),))
        self._recurse_attach(haplotype_B, offspring_id, ((breakpoint, self.genome_length),))

        self._apply_pending_bubbles()
        self._clear_modified_caches()

        return self._register_offspring(offspring_id)

    def recombine_multi(self, segments):
        self._pending_bubbles.clear()
        self._sync_to_grg()
        offspring_id = self.grg.make_node(negative=True)
        self._grow_node_arrays(self.grg.num_nodes - 1)
        self._gen_connected += 1

        # Bucket segments by parent so each parent gets one traversal
        # over the union of its contributed intervals. Avoids re-bubbling
        # the same ancestor when one parent contributes mutations to two
        # separated regions of the offspring's genome.
        by_parent = {}
        start = 0
        for parent_id, end in segments:
            if end > start:
                by_parent.setdefault(parent_id, []).append((start, end))
            start = end

        for parent_id, ivs in by_parent.items():
            ivs.sort()
            # Coalesce adjacent / overlapping intervals so the coverage
            # checks in `_recurse_attach` (which assume coalesced input)
            # remain correct. `recombination_intervals` produces strictly
            # alternating segments so adjacency is rare, but selfing or
            # odd breakpoint patterns can produce it.
            merged = []
            for L, R in ivs:
                if merged and L <= merged[-1][1]:
                    prev_L, prev_R = merged[-1]
                    merged[-1] = (prev_L, R if R > prev_R else prev_R)
                else:
                    merged.append((L, R))
            if self.debug_mode:
                print(f"BREAK parent={parent_id} intervals={merged}")
            self._recurse_attach(parent_id, offspring_id, tuple(merged))

        self._apply_pending_bubbles()
        self._clear_modified_caches()

        return self._register_offspring(offspring_id)

In [28]:
import pygrgl
from pygrgl import load_mutable_grg, grg_to_cyto_json
import re

try:
    from pygrgl.display import grg_to_cyto, DAG_STYLE
    from ipycytoscape import CytoscapeWidget
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    DAG_STYLE = None
    CytoscapeWidget = None

if 'NEGATIVE_NODE_IDS' not in globals():
    NEGATIVE_NODE_IDS = []

def display_grg(grg, title="GRG", negative_node_ids=None):
    """Display GRG - uses widget if available, otherwise text."""
    neg_ids = negative_node_ids if negative_node_ids is not None else NEGATIVE_NODE_IDS
    cyto = grg_to_cyto_json(grg, start_from=grg.get_root_nodes())

    for n in cyto['nodes']:
        node_id = int(n['data']['id'][1:])
        if node_id in neg_ids:
            disp = -(neg_ids.index(node_id) + 1)
            n['data']['label'] = n['data']['label'].replace(f'id={node_id}', f'id={disp}', 1)
        
        # Record positions of mutations for this node to display in label
        positions = []
        for mut_id in grg.get_mutations_for_node(node_id):
            mut = grg.get_mutation_by_id(mut_id)
            positions.append(mut.position)

        pattern = r"mutations\(\{.*?\}\)"
        positions_str = ", ".join(str(n) for n in positions)
        # Build the replacement string using your new contents
        replacement = f"mutations({{{positions_str}}})"

        # Replace the whole mutations({ ... }) with the new one
        new_str = re.sub(pattern, replacement, n['data']['label'])
        n['data']['label'] = new_str
        
    if WIDGETS_AVAILABLE:
        try:
            widget = CytoscapeWidget()
            widget.graph.add_graph_from_json(cyto, directed=True)
            widget.set_style(DAG_STYLE)
            widget.set_layout(name="dagre")
            display(widget)
            return
        except Exception as e:
            print(f"Widget display failed: {e}")
    
    print(f"\n{'='*50}")
    print(f" {title}")
    print(f"{'='*50}")
    print(f"Nodes: {len(cyto['nodes'])}, Edges: {len(cyto['edges'])}")
    print("\nNodes:")
    for n in cyto['nodes']:
        d = n['data']
        marker = "[S]" if d.get('is_sample') == 'True' else "   "
        print(f"  {marker} {d['id']}: {d['label']}")
    print("\nEdges (parent → child):")
    for e in cyto['edges']:
        src, tgt = e['data']['source'], e['data']['target']
        try:
            tgt_id = int(tgt[1:])
            tgt_display = f'n{-(neg_ids.index(tgt_id) + 1)}' if tgt_id in neg_ids else tgt
        except ValueError:
            tgt_display = tgt
        print(f"      {src} → {tgt_display}")
    print(f"{'='*50}\n")

grg = create_simple_grg()

def print_grg_state(grg, title="GRG State"):
    """Helper to print GRG state."""
    print(f"=== {title} ===")
    print(f"Nodes: {grg.num_nodes}, Edges: {grg.num_edges}, Mutations: {grg.num_mutations}")
    print(f"Samples: {grg.get_sample_nodes()}")
    print(f"Genome range: {grg.bp_range}")
    print()
    all_nodes = pygrgl.get_topo_order(grg, pygrgl.TraversalDirection.DOWN, grg.get_root_nodes())
    for node_id in all_nodes:
        display_id = -(NEGATIVE_NODE_IDS.index(node_id) + 1) if node_id in NEGATIVE_NODE_IDS else node_id
        up = grg.get_up_edges(node_id)
        down = grg.get_down_edges(node_id)
        muts = grg.get_mutations_for_node(node_id)
        mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
        is_sample = " [SAMPLE]" if grg.is_sample(node_id) else ""
        print(f"  Node {display_id}: parents={up}, children={down}, muts={mut_names}{is_sample}")

# Show initial state
#print_grg_state(grg, "BEFORE Recombination")
display_grg(grg, "BEFORE Recombination")

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [29]:
recomb = NonDuplicationRecombination(grg)

haplotype_A = 2  
haplotype_B = 3  
breakpoint = 3 
segments = []
segments.append((haplotype_A, breakpoint))
segments.append((haplotype_B, grg.bp_range[1]))

print("=" * 60)
print("RECOMBINATION")
print("=" * 60)
print(f"Haplotype A: {haplotype_A}")
print(f"Haplotype B: {haplotype_B}")
print(f"Breakpoint: {breakpoint}")
print()
print(f"Offspring inherits [0, {breakpoint}) from haplotype {haplotype_A}")
print(f"Offspring inherits [{breakpoint}, {grg.bp_range[1]}) from haplotype {haplotype_B}")
print()

# Perform recombination
offspring_id = recomb.recombine(haplotype_A, haplotype_B, breakpoint)

try:
    raw_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    current_samples = list(grg.get_sample_nodes())
    grg.set_samples(current_samples + [raw_id])
except AttributeError:
    pass

print(f"Created offspring node: {offspring_id}")
print()

# Show state after recombination
print_grg_state(grg, "AFTER Recombination")
display_grg(grg, "AFTER Recombination")

RECOMBINATION
Haplotype A: 2
Haplotype B: 3
Breakpoint: 3

Offspring inherits [0, 3) from haplotype 2
Offspring inherits [3, 11) from haplotype 3

Created offspring node: -1

=== AFTER Recombination ===
Nodes: 9, Edges: 10, Mutations: 11
Samples: [0, 1, 2, 8]
Genome range: (0, 11)

  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11']
  Node 6: parents=[], children=[0, 5], muts=['m9']
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8']
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7']
  Node 3: parents=[4], children=[8], muts=['m5']
  Node 8: parents=[3], children=[], muts=[] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [22]:
def verify_offspring_mutations(recomb, offspring_id, haplotype_A, haplotype_B, segments):
    """
    Verify that offspring has correct mutations based on recombination.
    
    Expected mutations:
    - From haplotype_A: all ancestral mutations with position < breakpoint
    - From haplotype_B: all ancestral mutations with position >= breakpoint
    """
    def get_all_ancestral_mutations(node_id, visited=None):
        """Get all mutation IDs from node and its ancestors."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return []
        visited.add(node_id)
        
        mutations = list(recomb.grg.get_mutations_for_node(node_id))
        
        for parent in recomb.grg.get_up_edges(node_id):
            mutations.extend(get_all_ancestral_mutations(parent, visited))
        
        return mutations
    
    def mut_name(mut_id):
        return f"m{mut_id + 1}"
    
    def get_position(mut_id):
        return recomb.grg.get_mutation_by_id(mut_id).position
    
    genome_length = recomb.grg.bp_range[1]
    
    # Get ancestral mutations for both haplotypes
    muts_A = get_all_ancestral_mutations(haplotype_A)
    muts_B = get_all_ancestral_mutations(haplotype_B)

    # Expected mutations for offspring based on segments
    expected_from_A = []
    expected_from_B = []

    # Determine expected mutations based on segments and their source haplotypes
    start = 0
    for parent, end in segments:
        if parent == haplotype_A:
            for m in muts_A:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_A.append(m)
        else:
            for m in muts_B:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_B.append(m)
        start = end
    
    expected_muts = set(expected_from_A + expected_from_B)
    expected_from_A.sort()
    expected_from_B.sort()
    
    # Get actual offspring mutations (traversing ancestry)
    grg_node_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1] if offspring_id < 0 else offspring_id
    actual_muts = set(get_all_ancestral_mutations(grg_node_id))

    muts_A.sort()
    muts_B.sort()
    
    if recomb.debug_mode:
        print("=== Mutation Verification ===")
        print(f"Haplotype A (node {haplotype_A}) mutations: {[get_position(m) for m in muts_A]}")
        print(f"Haplotype B (node {haplotype_B}) mutations: {[get_position(m) for m in muts_B]}")
        print()
        print(f"Expected from A: {[get_position(m) for m in expected_from_A]}")
        print(f"Expected from B: {[get_position(m) for m in expected_from_B]}")
        print()
        print(f"Expected: {sorted([get_position(m) for m in expected_muts])}")
        print(f"Actual:   {sorted([get_position(m) for m in actual_muts])}")
        print()
    
    if expected_muts == actual_muts:
        print("✓ Offspring mutations are CORRECT!")
        return True
    else:
        missing = expected_muts - actual_muts
        extra = actual_muts - expected_muts
        if missing:
            print(f"✗ Missing: {[get_position(m) for m in missing]}")
        if extra:
            print(f"✗ Extra: {[get_position(m) for m in extra]}")
        return False

# Verify the recombination
print("=== Verification ===")
breakpoints = []
breakpoints.append(breakpoint)
recombs = NonDuplicationRecombination(grg)
verify_offspring_mutations(recombs, offspring_id, haplotype_A, haplotype_B, segments)

=== Verification ===


IndexError: list index out of range

In [33]:
def generate_offspring(recomb, h1, h2, num_offspring, N=None):
    """
    Generate a recombined offspring from two parent haplotypes.
    
    Uses the recombination_intervals function to generate random
    crossover breakpoints, then applies non-duplication recombination.
    
    Args:
        grg: MutableGRG instance
        h1: First parent haplotype node ID
        h2: Second parent haplotype node ID
        N: Genome length (defaults to grg.bp_range[1])
        
    Returns:
        Tuple of (offspring_node_id, segments)
    """
    if N is None:
        N = grg.bp_range[1]
        print(f"Using genome length from GRG: {N}")
    
    offspring_ids = []
    for i in range(num_offspring):
        # Get recombination segments
        segments = recombination_intervals(h1, h2, N)
    
        # Perform recombination
        offspring_id = recomb.recombine_multi(segments)
        raw_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
        offspring_ids.append(raw_id)

        if recomb.debug_mode:
            print()
            print(f"Generated offspring node: {offspring_id}")
            print(f"Samples (including offspring): {grg.get_sample_nodes()}")
            print(f"Segments inherited: {segments}")
            print()

            print("Segment breakdown:")
            start = 0
            for parent, end in segments:
                print(f"  [{start}, {end}): from parent {parent}")
                start = end
            
            print("")
        verify_offspring_mutations(recomb, offspring_id, h1, h2, segments)

    # try:
    #     current_samples = list(grg.get_sample_nodes())
    #     raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    #     grg.set_samples(current_samples + [raw_id])
    # except AttributeError:
    #     pass  
    
    return segments, offspring_ids



print("=== Random Recombination Example ===")
grg_test = create_simple_grg() 

print(f"Loading: {GRG_FILE}")
print(f"Genome length: {grg_test.bp_range[1]}")
print(f"Initial samples: {grg_test.get_sample_nodes()}")
print()

display_grg(grg_test, "BEFORE random recombination")

genome = grg_test.bp_range
generations = 2

# Create recombination handler
recomb = NonDuplicationRecombination(grg_test)

for gen in range(generations):
    print(f"\n=== Generation {gen+1} ===")

    parent_indices = np.arange(len(grg_test.get_sample_nodes()))
    np.random.shuffle(parent_indices)
    new_offspring = []

    samples = grg_test.get_sample_nodes()
    shuffled_samples = np.random.shuffle(samples)

    #offspring, segs = generate_offspring(grg_test, h1=0, num_offspring=2, h2=3)
    for i in range(0, len(samples), 2):

        p1 = samples[i]
        p2 = samples[i+1]

        # display_grg(grg_test, f"BEFORE random recombination {i+1}")

        print(f"Selected parents for offspring {i+1}: h1={p1}, h2={p2}")
        segments, offspring_ids = generate_offspring(recomb, p1, p2, num_offspring=2, N=genome[1])
        new_offspring.extend(offspring_ids)

    new_offspring.sort()
    grg_test.set_samples(new_offspring)
    

# print()
# print(f"Generated offspring node: {offspring}")
# print(f"Samples (including offspring): {grg_test.get_sample_nodes()}")
# print(f"Segments inherited: {segs}")
# print()

# print("Segment breakdown:")
# start = 0
# for parent, end in segs:
#     print(f"  [{start}, {end}): from parent {parent}")
#     start = end

# Show after state
display_grg(grg_test, "AFTER random recombination")


=== Random Recombination Example ===
Loading: simple_example.grg
Genome length: 11
Initial samples: [0, 1, 2, 3]



CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…


=== Generation 1 ===
Selected parents for offspring 1: h1=2, h2=0
Generated breakpoints: [3 6 8]
✓ Offspring mutations are CORRECT!
Generated breakpoints: [1 6 8]
✓ Offspring mutations are CORRECT!
Selected parents for offspring 3: h1=3, h2=1
Generated breakpoints: [ 1  2 10]
✓ Offspring mutations are CORRECT!
Generated breakpoints: [ 1  7  8 10]
✓ Offspring mutations are CORRECT!

=== Generation 2 ===
Selected parents for offspring 1: h1=14, h2=8
Generated breakpoints: [1 3 5 6]
✓ Offspring mutations are CORRECT!
Generated breakpoints: [1 2 4 7]
✓ Offspring mutations are CORRECT!
Selected parents for offspring 3: h1=18, h2=11
Generated breakpoints: [3 4 5]
✓ Offspring mutations are CORRECT!
Generated breakpoints: [1 2 3 7]
✓ Offspring mutations are CORRECT!


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…